In [37]:
from pystac_client import Client

# Connecting to Sentinel-2
# Copernicus Data Space STAC catalog
STAC_URL = "https://stac.dataspace.copernicus.eu/v1"

catalog = Client.open(STAC_URL)

print("Connected to Copernicus!")

Connected to Copernicus!


In [38]:
# %%
# Load the plantation GeoJSON

import geopandas as gpd

plantation_gdf = gpd.read_file("../data/boundaries/PT.geojson")

print("Number of features:", len(plantation_gdf))
print(plantation_gdf[["field_3", "begin", "end", "Acreage"]])

Number of features: 2
             field_3  begin   end  Acreage
0  PT. PALMINA UTAMA   2216  2338  3711923
1  PT. PALMINA UTAMA   2216  2338  3711923


In [39]:
# %%
# Inspect plantation geometries

for i, feature in plantation_gdf.iterrows():
    print(f"\nFeature {i}")
    print("Plantation:", feature["field_3"])
    print("Geometry type:", feature.geometry.geom_type)
    print("Bounds:", feature.geometry.bounds)


Feature 0
Plantation: PT. PALMINA UTAMA
Geometry type: Polygon
Bounds: (111.5023, -0.1126, 111.5245, -0.08915)

Feature 1
Plantation: PT. PALMINA UTAMA
Geometry type: Polygon
Bounds: (111.3959, -0.04155, 111.4481, 0.031176)


In [40]:
# %%
# Select the first plantation block

plantation = plantation_gdf.iloc[0]

print("Plantation:", plantation["field_3"])
print("Area:", plantation["Acreage"])
print("Geometry:", plantation.geometry.geom_type)

Plantation: PT. PALMINA UTAMA
Area: 3711923
Geometry: Polygon


In [41]:
# %%
# Convert the plantation polygon to GeoJSON

plantation_geometry = plantation.geometry.__geo_interface__

print(plantation_geometry["type"])

Polygon


In [42]:
# %%
# Get bounding box from the actual plantation polygon

min_x, min_y, max_x, max_y = plantation.geometry.bounds

bbox = [
    min_x,
    min_y,
    max_x,
    max_y
]

print("Bounding box:")
print(bbox)

Bounding box:
[111.5023, -0.1126, 111.5245, -0.08915]


In [43]:
# %%
# Search Sentinel-2 images over the actual plantation

search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=plantation_geometry,
    datetime="2026-01-01/2026-09-09",
    query={
        "eo:cloud_cover": {
            "lte": 20
        }
    },
    max_items=100
)

items = list(search.items())

print(f"Found {len(items)} Sentinel-2 images")

Found 2 Sentinel-2 images


In [44]:
# %%
# Examine available Sentinel-2 images

for item in items:
    print(
        item.datetime.date(),
        round(item.properties.get("eo:cloud_cover", 999), 2),
        item.id
    )

2026-09-02 18.87 S2C_MSIL2A_20260902T024521_N0512_R132_T49MEV_20260902T073417
2026-08-08 1.13 S2B_MSIL2A_20260808T024529_N0512_R132_T49MEV_20260808T045442


In [45]:
# %%
# Load Sentinel Hub credentials

import os
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("SENTINELHUB_CLIENT_ID")
client_secret = os.getenv("SENTINELHUB_CLIENT_SECRET")

print("Client ID loaded:", client_id is not None)
print("Client Secret loaded:", client_secret is not None)

Client ID loaded: True
Client Secret loaded: True


In [46]:
# %%
# Authenticate with Copernicus

import requests

token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

response = requests.post(
    token_url,
    data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    },
)

print("Status code:", response.status_code)

if response.ok:
    token = response.json()["access_token"]
    print("Authentication successful!")
else:
    print("Authentication failed:")
    print(response.text)

Status code: 200
Authentication successful!
